In [1]:
import pandas as pd

### Leitura dos dados Master

In [2]:
dtype_map = {
    "homePhone": "string",
    "phone": "string",
    "tradeName": "string",
    "stateRegistration": "string",
    "isNewsletterOptIn": "string",  
    "businessPhone": "string",
    "corporateDocument": "string",
    "corporateName": "string",
}

df_master = pd.read_csv(
    r"C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_raw\masterdata_raw\Masterdata.csv",
    sep=";",
    dtype=dtype_map,
    low_memory=False
)

In [3]:
# -----------------------------
# 1) TIPAGEM (CUSTOMER)
# -----------------------------
TIPOS_CUSTOMER = {
    # strings (IDs, nomes, textos)
    "email": "string",
    "firstName": "string",
    "lastName": "string",
    "phone": "string",
    "homePhone": "string",
    "businessPhone": "string",
    "tradeName": "string",
    "stateRegistration": "string",
    "userId": "string",
    "document": "string",
    "localeDefault": "string",
    "attach": "string",
    "carttag": "string",
    "checkouttag": "string",
    "corporateDocument": "string",
    "corporateName": "string",
    "documentType": "string",
    "gender": "string",
    "visitedProductWithStockOutSkusTag": "string",
    "customerClass": "string",
    "priceTables": "string",
    "profilePicture": "string",
    "restrictions": "string",
    "tradePolicy": "string",
    "id": "string",
    "accountId": "string",
    "accountName": "string",
    "dataEntityId": "string",
    "createdBy": "string",
    "updatedBy": "string",
    "lastInteractionBy": "string",
    "tags": "string",
    "auto_filter": "string",
    "brandPurchasedTag": "string",
    "brandVisitedTag": "string",
    "categoryPurchasedTag": "string",
    "categoryVisitedTag": "string",
    "departmentVisitedTag": "string",
    "productPurchasedTag": "string",
    "productVisitedTag": "string",

    # números
    "rclastcartvalue": "Float64",
    "followers": "Int64",
    "birthDateMonth": "Int64",

    # booleanos
    "isCorporate": "boolean",
    "isNewsletterOptIn": "boolean",
    "approved": "boolean",

    # datas
    "birthDate": "datetime64[ns]",
    "rclastsessiondate": "datetime64[ns]",
    "createdIn": "datetime64[ns]",
    "updatedIn": "datetime64[ns]",
    "lastInteractionIn": "datetime64[ns]",
}

DT_COLS_C = [c for c, t in TIPOS_CUSTOMER.items() if t.startswith("datetime")]
INT_COLS_C = [c for c, t in TIPOS_CUSTOMER.items() if t == "Int64"]
FLOAT_COLS_C = [c for c, t in TIPOS_CUSTOMER.items() if t == "Float64"]
BOOL_COLS_C = [c for c, t in TIPOS_CUSTOMER.items() if t == "boolean"]
STR_COLS_C = [c for c, t in TIPOS_CUSTOMER.items() if t == "string"]


# -----------------------------
# 2) FUNÇÕES DE CONVERSÃO
# -----------------------------
def to_str(s: pd.Series) -> pd.Series:
    s = s.astype("string[python]").str.strip()
    s = s.replace({"": pd.NA, "NA": pd.NA, "NaN": pd.NA, "None": pd.NA})
    s = s.str.upper()  # opcional
    return s


def to_datetime(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    dt = pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)
    return dt.dt.strftime("%Y-%m-%d %H:%M:%S")


def to_number_str(s: pd.Series) -> pd.Series:
    s = s.astype("string[python]").str.strip()
    s = s.replace({"": pd.NA, "NA": pd.NA, "NaN": pd.NA, "None": pd.NA})
    s = s.str.replace(r"[^\d,\.\-]", "", regex=True)

    both = s.str.contains(",", na=False) & s.str.contains(r"\.", na=False)
    s = s.mask(both, s.str.replace(".", "", regex=False))

    s = s.str.replace(",", ".", regex=False)
    return s


def to_float(s: pd.Series) -> pd.Series:
    s = to_number_str(s)
    return pd.to_numeric(s, errors="coerce").astype("Float64")


def to_int(s: pd.Series) -> pd.Series:
    s = to_number_str(s)
    s = s.str.split(".", n=1).str[0]
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def to_bool(s: pd.Series) -> pd.Series:
    s = s.astype("string[python]").str.strip().str.upper()
    m = {
        "true": True, "false": False,
        "1": True, "0": False,
        "sim": True, "não": False, "nao": False,
        "s": True, "n": False,
        "verdadeiro": True, "falso": False,
        "t": True, "f": False,
        "ok": True
    }
    return s.map(m).astype("boolean")


# -----------------------------
# 3) APLICAR SCHEMA NO DF
# -----------------------------
def aplicar_schema_master(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    for c in STR_COLS_C:
        if c in df.columns:
            df[c] = to_str(df[c])

    for c in DT_COLS_C:
        if c in df.columns:
            df[c] = to_datetime(df[c])

    for c in FLOAT_COLS_C:
        if c in df.columns:
            df[c] = to_float(df[c])

    for c in INT_COLS_C:
        if c in df.columns:
            df[c] = to_int(df[c])

    for c in BOOL_COLS_C:
        if c in df.columns:
            df[c] = to_bool(df[c])

    return df

In [4]:
df_master = aplicar_schema_master(df_master)

In [5]:
col_rename_master = {
    "email": "EMAIL",
    "firstName": "FIRST_NAME",
    "homePhone": "HOME_PHONE",
    "lastName": "LAST_NAME",
    "phone": "PHONE",
    "isCorporate": "IS_CORPORATE",
    "tradeName": "TRADE_NAME",
    "rclastcart": "RC_LAST_CART",
    "rclastcartvalue": "RC_LAST_CART_VALUE",
    "rclastsession": "RC_LAST_SESSION",
    "rclastsessiondate": "RC_LAST_SESSION_DATE",
    "brandPurchasedTag": "BRAND_PURCHASED_TAG",
    "brandVisitedTag": "BRAND_VISITED_TAG",
    "categoryPurchasedTag": "CATEGORY_PURCHASED_TAG",
    "categoryVisitedTag": "CATEGORY_VISITED_TAG",
    "departmentVisitedTag": "DEPARTMENT_VISITED_TAG",
    "productPurchasedTag": "PRODUCT_PURCHASED_TAG",
    "productVisitedTag": "PRODUCT_VISITED_TAG",
    "stateRegistration": "STATE_REGISTRATION",
    "userId": "USER_ID",
    "document": "DOCUMENT",
    "isNewsletterOptIn": "IS_NEWSLETTER_OPT_IN",
    "localeDefault": "LOCALE_DEFAULT",
    "attach": "ATTACH",
    "approved": "APPROVED",
    "birthDate": "BIRTH_DATE",
    "businessPhone": "BUSINESS_PHONE",
    "carttag": "CART_TAG",
    "checkouttag": "CHECKOUT_TAG",
    "corporateDocument": "CORPORATE_DOCUMENT",
    "corporateName": "CORPORATE_NAME",
    "documentType": "DOCUMENT_TYPE",
    "gender": "GENDER",
    "visitedProductWithStockOutSkusTag": "VISITED_PRODUCT_STOCKOUT_SKUS_TAG",
    "customerClass": "CUSTOMER_CLASS",
    "priceTables": "PRICE_TABLES",
    "profilePicture": "PROFILE_PICTURE",
    "birthDateMonth": "BIRTH_DATE_MONTH",
    "restrictions": "RESTRICTIONS",
    "tradePolicy": "TRADE_POLICY",
    "id": "ID",
    "accountId": "ACCOUNT_ID",
    "accountName": "ACCOUNT_NAME",
    "dataEntityId": "DATA_ENTITY_ID",
    "createdBy": "CREATED_BY",
    "createdIn": "CREATED_IN",
    "updatedBy": "UPDATED_BY",
    "updatedIn": "UPDATED_IN",
    "lastInteractionBy": "LAST_INTERACTION_BY",
    "lastInteractionIn": "LAST_INTERACTION_IN",
    "followers": "FOLLOWERS",
    "tags": "TAGS",
    "auto_filter": "AUTO_FILTER",
}

In [6]:
df_master = df_master.rename(columns=col_rename_master)

In [7]:
df_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 508273 entries, 0 to 508272
Data columns (total 53 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   EMAIL                              508273 non-null  string 
 1   FIRST_NAME                         249239 non-null  string 
 2   HOME_PHONE                         247601 non-null  string 
 3   LAST_NAME                          249146 non-null  string 
 4   PHONE                              21067 non-null   string 
 5   IS_CORPORATE                       0 non-null       boolean
 6   TRADE_NAME                         163 non-null     string 
 7   RC_LAST_CART                       338343 non-null  str    
 8   RC_LAST_CART_VALUE                 348020 non-null  Float64
 9   RC_LAST_SESSION                    399246 non-null  str    
 10  RC_LAST_SESSION_DATE               259518 non-null  str    
 11  BRAND_PURCHASED_TAG                378556 non-null

In [77]:
df_master.to_csv(r'C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_enriched\masterdata_enriched\Masterdata_enriched.csv', index=False)

### Leitura dos dados Pedidos

In [20]:
dtype_map = {
    "Phone": "string",
    "Cancelled By": "string",
    "Cancellation Reason": "string",
}

df_jan_jun = pd.read_csv(
    r"C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_raw\pedidos_raw\Jan-Jun 2025.csv",
    sep=";",
    dtype=dtype_map,
    low_memory=False
)

In [21]:
dtype_map = {
    "Corporate Name": "string",
    "Delivered": "string",
}

df_jul_dez = pd.read_csv(
    r"C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_raw\pedidos_raw\Jul-Dez 2025.csv",
    sep=";",
    dtype=dtype_map,
    low_memory=False
)

In [22]:
df_jan = pd.read_csv(
    r"C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_raw\pedidos_raw\Jan 2026.csv",
    sep=";")

In [23]:
df_f = pd.concat([df_jan_jun, df_jul_dez, df_jan], ignore_index=True)

### Concatenação dos Pedidos

In [24]:
# Evita conflitos com backend pyarrow em string
pd.options.mode.string_storage = "python"

# -----------------------------
# 1) TIPOS POR COLUNA (94)
# -----------------------------
TIPOS = {
    # strings (IDs, nomes, textos)
    "Origin": "string",
    "Order": "string",
    "Client Name": "string",
    "Client Last Name": "string",
    "Client Document": "string",
    "Email": "string",
    "Phone": "string",
    "UF": "string",
    "City": "string",
    "Address Identification": "string",
    "Address Type": "string",
    "Receiver Name": "string",
    "Street": "string",
    "Number": "string",
    "Complement": "string",
    "Neighborhood": "string",
    "Reference": "string",
    "Postal Code": "string",
    "SLA Type": "string",
    "Courrier": "string",
    "Delivery Deadline": "string",
    "Status": "string",
    "UtmMedium": "string",
    "UtmSource": "string",
    "UtmCampaign": "string",
    "Coupon": "string",
    "Payment System Name": "string",
    "ID_SKU": "string",
    "Category Ids Sku": "string",
    "Reference Code": "string",
    "SKU Name": "string",
    "SKU Path": "string",
    "Item Attachments": "string",
    "List Id": "string",
    "List Type Name": "string",
    "Discounts Names": "string",
    "Call Center Email": "string",
    "Call Center Code": "string",
    "Tracking Number": "string",
    "Host": "string",
    "GiftRegistry ID": "string",
    "Seller Name": "string",
    "Status TimeLine": "string",
    "Obs": "string",
    "UtmiPart": "string",
    "UtmiCampaign": "string",
    "UtmiPage": "string",
    "Seller Order Id": "string",
    "Acquirer": "string",
    "Authorization Id": "string",
    "TID": "string",
    "NSU": "string",
    "Card First Digits": "string",
    "Card Last Digits": "string",
    "Payment Approved By": "string",
    "Cancelled By": "string",
    "Cancellation Reason": "string",
    "Gift Card Name": "string",
    "Gift Card Caption": "string",
    "Corporate Name": "string",
    "Corporate Document": "string",
    "TransactionId": "string",
    "PaymentId": "string",
    "PaymentOrigin": "string",
    "SalesChannel": "string",
    "marketingTags": "string",
    "Currency Code": "string",
    "Invoice Numbers": "string",
    "Country": "string",
    "Input Invoices Numbers": "string",
    "Output Invoices Numbers": "string",
    "Status raw value (temporary)": "string",

    # inteiros
    "Sequence": "Int64",
    "Installments": "Int64",
    "Quantity_SKU": "Int64",

    # números (valores)
    "Payment Value": "Float64",
    "SKU Value": "Float64",
    "SKU Selling Price": "Float64",
    "SKU Total Price": "Float64",
    "Service (Price/ Selling Price)": "Float64",
    "Shipping List Price": "Float64",
    "Shipping Value": "Float64",
    "Total Value": "Float64",
    "Discounts Totals": "Float64",
    "SKU RewardValue": "Float64",
    "Taxes": "Float64",

    # booleanos
    "Delivered": "boolean",
    "Is Marketplace cetified": "boolean",
    "Is Checked In": "boolean",

    # datas
    "Creation Date": "datetime64[ns]",
    "Estimate Delivery Date": "datetime64[ns]",
    "Last Change Date": "datetime64[ns]",
    "Authorized Date": "datetime64[ns]",
    "Cancellation Data": "datetime64[ns]",
}

DT_COLS = [c for c, t in TIPOS.items() if t.startswith("datetime")]
INT_COLS = [c for c, t in TIPOS.items() if t == "Int64"]
FLOAT_COLS = [c for c, t in TIPOS.items() if t == "Float64"]
BOOL_COLS = [c for c, t in TIPOS.items() if t == "boolean"]
STR_COLS = [c for c, t in TIPOS.items() if t == "string"]


# -----------------------------
# 2) FUNÇÕES DE CONVERSÃO
# -----------------------------
def to_str(s: pd.Series) -> pd.Series:
    s = s.astype("string[python]").str.strip()
    s = s.replace({"": pd.NA, "NA": pd.NA, "NaN": pd.NA, "None": pd.NA})
    s = s.str.upper()  # opcional: padroniza para maiúsculas
    return s

def to_datetime(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()  
    dt = pd.to_datetime(s, utc=True, errors="coerce")  
    return dt.dt.tz_convert(None)  


def to_number_str(s: pd.Series) -> pd.Series:
    s = to_str(s)
    # remove qualquer coisa que não seja número, vírgula, ponto ou sinal
    s = s.str.replace(r"[^\d,\.\-]", "", regex=True)

    # se tiver vírgula e ponto, assume . milhar e , decimal -> remove pontos e troca vírgula por ponto
    both = s.str.contains(",", na=False) & s.str.contains(r"\.", na=False)
    s = s.mask(both, s.str.replace(".", "", regex=False))

    # troca vírgula por ponto para parse numérico
    s = s.str.replace(",", ".", regex=False)

    return s


def to_float(s: pd.Series) -> pd.Series:
    s = to_number_str(s)
    return pd.to_numeric(s, errors="coerce").astype("Float64")


def to_int(s: pd.Series) -> pd.Series:
    s = to_number_str(s)
    # corta decimal se existir
    s = s.str.split(".", n=1).str[0]
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def to_bool(s: pd.Series) -> pd.Series:
    s = to_str(s).str.upper()
    m = {
        "true": True, "false": False,
        "1": True, "0": False,
        "sim": True, "não": False, "nao": False,
        "s": True, "n": False,
        "verdadeiro": True, "falso": False,
        "t": True, "f": False,
        "ok": True
    }
    return s.map(m).astype("boolean")


# -----------------------------
# 3) APLICAR SCHEMA NO DF
# -----------------------------
def aplicar_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # garante nomes exatamente (tira espaços extras)
    df.columns = [c.strip() for c in df.columns]

    for c in STR_COLS:
        if c in df.columns:
            df[c] = to_str(df[c])

    for c in DT_COLS:
        if c in df.columns:
            df[c] = to_datetime(df[c])

    for c in FLOAT_COLS:
        if c in df.columns:
            df[c] = to_float(df[c])

    for c in INT_COLS:
        if c in df.columns:
            df[c] = to_int(df[c])

    for c in BOOL_COLS:
        if c in df.columns:
            df[c] = to_bool(df[c])

    return df

In [25]:
df_f = aplicar_schema(df_f)

In [26]:
col_rename = {
    "Origin": "ORIGIN",
    "Order": "ORDER_ID",
    "Sequence": "SEQUENCE",
    "Creation Date": "CREATION_DATE",
    "Client Name": "CLIENT_FIRST_NAME",
    "Client Last Name": "CLIENT_LAST_NAME",
    "Client Document": "CLIENT_DOCUMENT",
    "Email": "EMAIL",
    "Phone": "PHONE",
    "UF": "STATE_UF",
    "City": "CITY",
    "Address Identification": "ADDRESS_ID",
    "Address Type": "ADDRESS_TYPE",
    "Receiver Name": "RECEIVER_NAME",
    "Street": "STREET",
    "Number": "ADDRESS_NUMBER",
    "Complement": "ADDRESS_COMPLEMENT",
    "Neighborhood": "NEIGHBORHOOD",
    "Reference": "ADDRESS_REFERENCE",
    "Postal Code": "POSTAL_CODE",
    "SLA Type": "SLA_TYPE",
    "Courrier": "COURIER",
    "Estimate Delivery Date": "ESTIMATED_DELIVERY_DATE",
    "Delivery Deadline": "DELIVERY_DEADLINE",
    "Status": "STATUS",
    "Last Change Date": "LAST_CHANGE_DATE",
    "UtmMedium": "UTM_MEDIUM",
    "UtmSource": "UTM_SOURCE",
    "UtmCampaign": "UTM_CAMPAIGN",
    "Coupon": "COUPON",
    "Payment System Name": "PAYMENT_SYSTEM_NAME",
    "Installments": "INSTALLMENTS",
    "Payment Value": "PAYMENT_VALUE",
    "Quantity_SKU": "SKU_QUANTITY",
    "ID_SKU": "SKU_ID",
    "Category Ids Sku": "SKU_CATEGORY_IDS",
    "Reference Code": "REFERENCE_CODE",
    "SKU Name": "SKU_NAME",
    "SKU Value": "SKU_VALUE",
    "SKU Selling Price": "SKU_SELLING_PRICE",
    "SKU Total Price": "SKU_TOTAL_PRICE",
    "SKU Path": "SKU_PATH",
    "Item Attachments": "ITEM_ATTACHMENTS",
    "List Id": "LIST_ID",
    "List Type Name": "LIST_TYPE_NAME",
    "Service (Price/ Selling Price)": "SERVICE_PRICE_SELLING_PRICE",
    "Shipping List Price": "SHIPPING_LIST_PRICE",
    "Shipping Value": "SHIPPING_VALUE",
    "Total Value": "TOTAL_VALUE",
    "Discounts Totals": "DISCOUNTS_TOTAL",
    "Discounts Names": "DISCOUNTS_NAMES",
    "Call Center Email": "CALL_CENTER_EMAIL",
    "Call Center Code": "CALL_CENTER_CODE",
    "Tracking Number": "TRACKING_NUMBER",
    "Host": "HOST",
    "GiftRegistry ID": "GIFT_REGISTRY_ID",
    "Seller Name": "SELLER_NAME",
    "Status TimeLine": "STATUS_TIMELINE",
    "Obs": "OBS",
    "UtmiPart": "UTMI_PART",
    "UtmiCampaign": "UTMI_CAMPAIGN",
    "UtmiPage": "UTMI_PAGE",
    "Seller Order Id": "SELLER_ORDER_ID",
    "Acquirer": "ACQUIRER",
    "Authorization Id": "AUTHORIZATION_ID",
    "TID": "TID",
    "NSU": "NSU",
    "Card First Digits": "CARD_FIRST_DIGITS",
    "Card Last Digits": "CARD_LAST_DIGITS",
    "Payment Approved By": "PAYMENT_APPROVED_BY",
    "Cancelled By": "CANCELLED_BY",
    "Cancellation Reason": "CANCELLATION_REASON",
    "Gift Card Name": "GIFT_CARD_NAME",
    "Gift Card Caption": "GIFT_CARD_CAPTION",
    "Authorized Date": "AUTHORIZED_DATE",
    "Corporate Name": "CORPORATE_NAME",
    "Corporate Document": "CORPORATE_DOCUMENT",
    "TransactionId": "TRANSACTION_ID",
    "PaymentId": "PAYMENT_ID",
    "PaymentOrigin": "PAYMENT_ORIGIN",
    "SalesChannel": "SALES_CHANNEL",
    "marketingTags": "MARKETING_TAGS",
    "Delivered": "DELIVERED",
    "SKU RewardValue": "SKU_REWARD_VALUE",
    "Is Marketplace cetified": "IS_MARKETPLACE_CERTIFIED",
    "Is Checked In": "IS_CHECKED_IN",
    "Currency Code": "CURRENCY_CODE",
    "Taxes": "TAXES",
    "Invoice Numbers": "INVOICE_NUMBERS",
    "Country": "COUNTRY",
    "Input Invoices Numbers": "INPUT_INVOICE_NUMBERS",
    "Output Invoices Numbers": "OUTPUT_INVOICE_NUMBERS",
    "Status raw value (temporary)": "STATUS_RAW_VALUE_TEMP",
    "Cancellation Data": "CANCELLATION_DATA",
}

In [27]:
df_f = df_f.rename(columns=col_rename)

In [28]:
df_f.to_csv(r"C:\Users\igor.pedro\Downloads\Igor_feng\to_do\ecommerce_flamengo\data_enriched\pedidos_enriched\pedidos_enriched.csv", index=False)